# Anomaly Detection Results Visualization

## Purpose of This Notebook

This notebook performs **post-processing and analysis** of the anomaly detection pipeline output.

Its goal is to:

- load anomaly detection results produced by the pipeline,
- aggregate them over time,
- evaluate detection performance,
- and visualize the results in a clear and interpretable form.

This notebook represents the **analysis and reporting stage** of the system.

---

## Role in the Overall Pipeline

This notebook operates **outside of the real-time pipeline**.

Important distinction:

- The producer and consumer notebooks are part of the **event-driven system**.
- This notebook works on **persisted results**.

This separation reflects real security workflows, where:
- detection pipelines run continuously,
- analysis and reporting are performed on stored data.

---

## Key Architectural Ideas Demonstrated

### 1. Separation of Processing and Analysis

The pipeline:
- produces structured data,
- does not generate dashboards.

This notebook:
- consumes stored results,
- performs analytical aggregation,
- generates visual representations.

This design avoids coupling analytics logic to the pipeline itself.

---

### 2. Time-Based Aggregation

Security analysis is inherently **time-oriented**.

In this notebook:
- raw event timestamps are grouped into fixed time windows,
- anomaly counts are aggregated per time bucket.

This enables detection of:
- bursts of anomalous activity,
- changes in anomaly patterns over time,
- trends in pipeline output.

---

### 3. Performance Evaluation

The notebook evaluates the anomaly detection performance by comparing:

- Predicted anomalies vs. ground truth
- Confusion matrix analysis
- Precision, recall, and F1-score metrics

This allows assessment of the model's effectiveness in the real-time pipeline.

---

## Input Data

The notebook reads data from: `anomaly_detection_results.tsv`

This file was produced by the **consumer/anomaly detector stage**.

It contains:
- event metadata,
- anomaly scores and predictions,
- ground truth labels (for evaluation),
- timestamps for temporal analysis.

---

## Visualizations

The notebook generates several visualizations:

1. **Time Series of Anomalies**: Shows anomaly counts over time
2. **Anomaly Score Distribution**: Histogram of anomaly scores
3. **Confusion Matrix**: Evaluation of prediction accuracy
4. **Feature Distributions**: Comparison of normal vs. anomalous events
5. **Performance Metrics**: Precision, recall, F1-score over time

---

## What This Notebook Does NOT Do

- It does **not** modify the pipeline.
- It does **not** send data back to Kafka.
- It does **not** emit traces.
- It does **not** perform real-time monitoring.

Its purpose is **offline analysis and interpretation**.

---

## How This Notebook Is Used in the Lab

1. Run the producer and consumer notebooks for several minutes.
2. Allow `anomaly_detection_results.tsv` to accumulate data.
3. Stop or pause the pipeline if needed.
4. Run this notebook to analyze the collected results.
5. Experiment with:
   - different time bucket sizes,
   - different aggregation dimensions,
   - various evaluation metrics.

---

## Suggested Experiments

You are encouraged to modify this notebook to explore:

- filtering by specific users or devices,
- different time resolutions (e.g. 30 seconds, 5 minutes),
- correlation analysis between features and anomaly scores,
- threshold optimization for different use cases.

---

## Key Takeaway

> **Anomaly detection pipelines produce data — insight comes from analysis.**

Understanding how to interpret pipeline output is as important
as understanding how the pipeline itself is built.

This notebook closes the loop:
from raw events → anomaly detection → structured results → insight.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Set the style
sns.set_theme(style="whitegrid")

TSV_PATH = "anomaly_detection_results.tsv"
df = pd.read_csv(TSV_PATH, sep='\t')

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["time_bucket"] = df["timestamp"].dt.floor("1min")

print(f"Loaded {len(df)} records from {TSV_PATH}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Anomaly rate (predicted): {df['is_anomaly_predicted'].mean()*100:.1f}%")
print(f"Anomaly rate (ground truth): {df['is_anomaly_ground_truth'].mean()*100:.1f}%")

FileNotFoundError: [Errno 2] No such file or directory: 'anomaly_detection_results.tsv'

In [ ]:
# Time series of anomalies
plt.figure(figsize=(18, 8))

# Aggregate anomalies over time
anomaly_counts = (
    df.groupby(["time_bucket", "is_anomaly_predicted"])
      .size()
      .unstack(fill_value=0)
      .sort_index()
)

anomaly_counts.columns = ['Normal', 'Anomaly']
anomaly_counts.plot(
    kind="bar",
    stacked=True,
    width=0.8,
    color=['lightblue', 'red'],
    ax=plt.gca()
)

plt.title("QR Scan Anomalies Over Time")
plt.xlabel("Time")
plt.ylabel("Number of Events")
plt.legend(title="Prediction", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Anomaly score distribution
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(data=df, x='anomaly_score', bins=50, kde=True)
plt.axvline(x=0, color='r', linestyle='--', label='Threshold')
plt.title('Distribution of Anomaly Scores')
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(data=df, x='anomaly_score', hue='is_anomaly_predicted', bins=50, kde=True)
plt.axvline(x=0, color='r', linestyle='--', label='Threshold')
plt.title('Anomaly Scores by Prediction')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix and performance metrics
if 'is_anomaly_ground_truth' in df.columns:
    y_true = df['is_anomaly_ground_truth']
    y_pred = df['is_anomaly_predicted']
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Anomaly'], 
                yticklabels=['Normal', 'Anomaly'])
    plt.title('Confusion Matrix')
    plt.ylabel('Ground Truth')
    plt.xlabel('Predicted')
    plt.show()
    
    # Classification report
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))
    
    # Performance over time
    df['correct'] = (y_true == y_pred).astype(int)
    
    performance_over_time = (
        df.groupby("time_bucket")
          .agg({
              'correct': 'mean',
              'is_anomaly_predicted': 'sum',
              'is_anomaly_ground_truth': 'sum'
          })
          .sort_index()
    )
    
    plt.figure(figsize=(18, 6))
    
    plt.subplot(1, 3, 1)
    performance_over_time['correct'].plot(kind='line', marker='o')
    plt.title('Accuracy Over Time')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 3, 2)
    performance_over_time[['is_anomaly_predicted', 'is_anomaly_ground_truth']].plot(kind='bar', ax=plt.gca())
    plt.title('Anomalies Detected vs Ground Truth')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 3, 3)
    (performance_over_time['is_anomaly_predicted'] - performance_over_time['is_anomaly_ground_truth']).plot(kind='bar')
    plt.title('Detection Error (Predicted - Ground Truth)')
    plt.ylabel('Error')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature distributions for normal vs anomalous events
features_to_plot = ['url_length', 'dot_count', 'hour']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, feature in enumerate(features_to_plot):
    sns.histplot(data=df, x=feature, hue='is_anomaly_predicted', 
                 bins=30, kde=True, ax=axes[i], palette=['lightblue', 'red'])
    axes[i].set_title(f'{feature.replace("_", " ").title()} Distribution')
    axes[i].legend(title='Predicted', labels=['Normal', 'Anomaly'])

plt.tight_layout()
plt.show()

# Protocol distribution
plt.figure(figsize=(8, 6))
protocol_counts = pd.crosstab(df['protocol'], df['is_anomaly_predicted'], normalize='index')
protocol_counts.plot(kind='bar', stacked=True, color=['lightblue', 'red'])
plt.title('Protocol Distribution by Prediction')
plt.ylabel('Proportion')
plt.legend(title='Predicted', labels=['Normal', 'Anomaly'])
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Summary statistics
print("Summary Statistics:")
print(f"Total events processed: {len(df)}")
print(f"Time span: {df['timestamp'].max() - df['timestamp'].min()}")
print(f"Average events per minute: {len(df) / ((df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 60):.1f}")

if 'is_anomaly_ground_truth' in df.columns:
    print(f"\nGround truth anomaly rate: {df['is_anomaly_ground_truth'].mean()*100:.1f}%")
    print(f"Predicted anomaly rate: {df['is_anomaly_predicted'].mean()*100:.1f}%")
    print(f"Detection accuracy: {(df['is_anomaly_ground_truth'] == df['is_anomaly_predicted']).mean()*100:.1f}%")

print(f"\nAnomaly score statistics:")
print(df['anomaly_score'].describe())

print(f"\nTop anomalous URLs:")
anomalous_events = df[df['is_anomaly_predicted'] == 1].sort_values('anomaly_score')
print(anomalous_events[['url', 'anomaly_score', 'url_length', 'dot_count']].head(10))